# Translation Corrector

## About
This notebook validates the the German and English translations.

There are two phases:

- the first phase just checks the structural aspects of the AsciiDoc files. It makes life so easier if there are the same amount of text lines and blank lines for each language section. The result are hint where the structure of one translation doesn't match the other.
- the second phase checks for content and differences between the translations. The result is a assessment sheet as well as merge files that include corrections. Note that this makes only sense to use if the files are structural identical because the translation checks on a line-by-line basis.

## Prerequisites




### Load all relevant files

In [1]:
import glob

# get all AsciiDoc files (hack: starting with "0", doesn't support more chapters > 9)
all_adoc_files = glob.glob('../docs/0*/*.adoc', recursive=True)

# filter out
# - "00-introduction" because they are just the files for combining AsciiDoc files
# - the reference list with books where translation validation makes no sense
file_list = [f for f in all_adoc_files if not f.endswith('00-introduction.adoc') and not f.endswith('00-references.adoc')]

print(f"{len(file_list)} files found.")
file_list[:5] # just show first 5 items to not clutter the notebook with too much text

55 files found.


['../docs\\00-preamble\\01-what-to-expect.adoc',
 '../docs\\00-preamble\\02-out-of-scope.adoc',
 '../docs\\00-preamble\\03-prerequisites.adoc',
 '../docs\\00-preamble\\04-structure-timing-didactics.adoc',
 '../docs\\00-preamble\\05-exam-relevance-levels.adoc']

### Helper function: show output as HTML

This function used at the end of a cell will render the later included links accordingly


In [48]:
from notebook import notebookapp
from IPython.display import display, HTML

def show(df):
    return display(HTML(df.to_html(escape=False)))

### Import files into a pandas DataFrames

In [103]:
import pandas as pd



import os

df_list = []

# make a Dataframe for each file
for file in file_list:

    # read in the AsciiDoc file, keep the blank lines for the line numbers
    df = pd.read_csv(file, names=['text'], sep="\r", skip_blank_lines=False)

     # add file name as first column
    df.insert(0,'filename',file)

    # Mark the rows with language tags
    df.loc[df['text'] == '// tag::DE[]', 'lang'] = 'DE'
    df.loc[df['text'] == '// tag::EN[]', 'lang'] = 'EN'

    # Forward fill the 'lang' column to propagate the tags to the subsequent rows
    df['lang'] = df['lang'].ffill()

    # add line numbers in file
    df['line'] = df.index + 1

    df_de = df[df['lang'] == 'DE'].reset_index(drop=True)
    df_en = df[df['lang'] == 'EN'].reset_index(drop=True)
    
    # Create Side-by-Side (sbs) Dataframe
    sbs_df = df_de[['filename', 'text', 'line']].join(df_en[['text', 'line']], lsuffix="_DE", rsuffix="_EN")

    # Enrich Dataframe with direct links into file in VS Code via HTML
    # Maybe the `replace` at the end needs to be adjusted to whatever IDE and operating system you have, but the default should work fine for most environments
    urls_DE = sbs_df.apply(lambda x: f"<a href=\"vscode://file/{os.path.abspath(x['filename'])}:{float(x['line_DE'])}\">{float(x['line_DE'])}</a>".replace("/mnt/c/", "C:/"), axis=1).apply(lambda x: f"{x}")
    urls_EN = sbs_df.apply(lambda x: f"<a href=\"vscode://file/{os.path.abspath(x['filename'])}:{float(x['line_EN'])}\">{float(x['line_EN'])}</a>".replace("/mnt/c/", "C:/"), axis=1).apply(lambda x: f"{x}")
    sbs_df.insert(sbs_df.columns.get_loc('line_DE') + 1, 'link_DE', urls_DE)
    sbs_df.insert(sbs_df.columns.get_loc('line_EN') + 1, 'link_EN', urls_EN)

    df_list.append(sbs_df)

structural_df = pd.concat(df_list, ignore_index=True)
show(structural_df.head())

,filename,text_DE,line_DE,link_DE,text_EN,line_EN,link_EN
0,../docs\00-preamble\01-what-to-expect.adoc,// tag::DE[],1.0,1.0,// tag::EN[],36.0,36.0
1,../docs\00-preamble\01-what-to-expect.adoc,=== Was dieser Lehrplan enthält,2.0,2.0,=== What Does this Curriculum Contain,37.0,37.0
2,../docs\00-preamble\01-what-to-expect.adoc,"Dieser Lehrplan für den _Certified Professional for Software Architecture - Foundation Level_ (CPSA-F) beinhaltet die Lernziele, die man beherrschen sollte, um die Rolle Softwarearchitekt:in zu übernehmen.",3.0,3.0,This curriculum for the Certified Professional for Software Architecture – Foundation Level (CPSA-F) outlines the essential learning goals that should be mastered to take up the role of software architect.,38.0,38.0
3,../docs\00-preamble\01-what-to-expect.adoc,NaN,4.0,4.0,NaN,39.0,39.0
4,../docs\00-preamble\01-what-to-expect.adoc,Seine Struktur orientiert sich an den grundlegenden Aktivitäten und Verantwortlichkeiten der Softwarearchitektur als Rolle:,5.0,5.0,It is structured along the fundamental activities and responsibilities of software architecture as a role:,40.0,40.0


## Outputs

### Learning Goals

In [138]:
learning_goals = structural_df[structural_df['text_DE'].fillna('').str.startswith("==== LZ")]['text_DE']
learning_goals = learning_goals.str.replace("==== ", "").replace("\ \[.*\]", "", regex=True)
learning_goals.head()

207    LZ 01-01: Definitionen von Softwarearchitektur...
220    LZ 01-02: Ziele und Nutzen von Softwarearchite...
235    LZ 01-03: Aufgaben und Verantwortung von Softw...
257    LZ 01-04: Abgrenzung zu anderen Architekturdom...
277    LZ 01-05: Rolle von Softwarearchitekt:innen mi...
Name: text_DE, dtype: object

### R1 Learning Goals

In [139]:
learning_goals_r1 = learning_goals[learning_goals.str.contains("R1")]
learning_goals_r1

207    LZ 01-01: Definitionen von Softwarearchitektur...
220    LZ 01-02: Ziele und Nutzen von Softwarearchite...
235    LZ 01-03: Aufgaben und Verantwortung von Softw...
277    LZ 01-05: Rolle von Softwarearchitekt:innen mi...
334     LZ 02-01: Stakeholder-Anliegen verstehen (R1-R3)
374    LZ 02-02: Anforderungen und Randbedingungen kl...
418    LZ 02-03: Qualitäten eines Softwaresystems ver...
448    LZ 02-04: Anforderungen an Qualitäten formulie...
461    LZ 02-05: Explizite Aussagen vor impliziten An...
514    LZ 03-01: Anforderungen durch Architektur erre...
527       LZ 03-02: Softwarearchitekturen entwerfen (R1)
545    LZ 03-03: Vorgehen und Heuristiken zur Archite...
561    LZ 03-04: Entwurfsprinzipien erläutern und anw...
616    LZ 03-05: Zusammenhang zwischen Feedback-Schle...
632    LZ 03-06: Abhängigkeiten von Bausteinen manage...
659    LZ 03-07: Schnittstellen entwerfen und festleg...
681    LZ 03-08: Wichtige Architekturmuster beschreib...
740    LZ 03-10: Querschnittsth

## Cleaning Phase I: Structural checking


### Check for different blank lines

Check which lines in the German version are blank but not so in the English version.

You can delete the `head()` method in the line before the last line for showing all findings.

Tip: Work backwards from the bottom to the top when you want to fix this manally.

Pro-Tip: Introduce a automated formatting for AsciiDoc file or write a tool (if existent)

In [104]:
# Filter rows where both text_DE and text_EN are not NaN at the same time
blanks_df = structural_df[(structural_df['text_DE'].isna() & ~structural_df['text_EN'].isna())]

print(f"{len(blanks_df)} differences regarding lines found!")
# Display the filtered DataFrame
# hint: remove the `.head()` method to get a complete list
display(HTML(blanks_df.head().to_html(escape=False)))

116 differences regarding lines found!


,filename,text_DE,line_DE,link_DE,text_EN,line_EN,link_EN
12,../docs\00-preamble\01-what-to-expect.adoc,NaN,13.0,13.0,=== What Does a Foundation Level Training Convey?,48.0,48.0
19,../docs\00-preamble\01-what-to-expect.adoc,NaN,20.0,20.0,"*\tdiscuss and reconcile fundamental architectural decisions with stakeholders from requirements, management, development, operations and test",55.0,55.0
21,../docs\00-preamble\01-what-to-expect.adoc,NaN,22.0,22.0,"*\tdocument and communicate software architectures based upon architectural views, architecture patterns and technical concepts.",57.0,57.0
25,../docs\00-preamble\01-what-to-expect.adoc,NaN,26.0,26.0,* the term software architecture and its meaning,61.0,61.0
27,../docs\00-preamble\01-what-to-expect.adoc,NaN,28.0,28.0,* the roles of software architects within development projects,63.0,63.0


### Check for wrong abbreviations

In [105]:
ev_de_check = structural_df[structural_df['text_DE'].fillna('').str.contains("e\.V\.")]
display(HTML(ev_de_check.to_html(escape=False)))

,filename,text_DE,line_DE,link_DE,text_EN,line_EN,link_EN
37,../docs\00-preamble\02-out-of-scope.adoc,Dieser Lehrplan reflektiert den aus heutiger Sicht des iSAQB e.V. notwendigen und sinnvollen Inhalt zur Erreichung der Lernziele des CPSA-F. Er stellt keine vollständige Beschreibung des Wissensgebiets „Softwarearchitektur“ dar.,5.0,5.0,This curriculum reflects the contents currently considered by the iSAQB members to be necessary and useful for achieving the learning goals of CPSA-F. It is not a comprehensive description of the entire domain of 'software architecture'.,24.0,24.0


In [106]:
ev_en_check = structural_df[structural_df['text_EN'].fillna('').str.contains("e\.V\.")]
display(HTML(ev_en_check.to_html(escape=False)))

,filename,text_DE,line_DE,link_DE,text_EN,line_EN,link_EN
46,../docs\00-preamble\02-out-of-scope.adoc,"* Test (siehe dazu das Ausbildungs- und Zertifizierungsprogramm des ISTQB e. V., https://istqb.org, International Software Testing Qualification Board)",14.0,14.0,"* software testing (please refer to the education and certification program by ISTQB e.V., https://istqb.org, International Software Testing Qualification Board)",33.0,33.0


### Check for different amount of sentences in a line

In [107]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)

sentence_de_count = lambda x : len(sent_tokenize(x, "german"))
sentence_en_count = lambda x : len(sent_tokenize(x, "english"))


structural_df['sentence_count_DE'] = structural_df['text_DE'].fillna('').apply(sentence_de_count)
structural_df['sentence_count_EN'] = structural_df['text_EN'].fillna('').apply(sentence_en_count)

different_amount_of_sentences_df = structural_df[structural_df['sentence_count_DE'] != structural_df['sentence_count_EN']]

print(f"{len(different_amount_of_sentences_df)} differences regarding amount of sentences found!")

# just display the first finding per file because the other findings might be subsequent errors
sentences_df_grouped = different_amount_of_sentences_df.groupby("filename").first()

print("\nFirst errors per file:")
display(HTML(sentences_df_grouped.to_html(escape=False)))

283 differences regarding amount of sentences found!

First errors per file:


,text_DE,line_DE,link_DE,text_EN,line_EN,link_EN,sentence_count_DE,sentence_count_EN
filename,,,,,,,,
../docs\00-preamble\01-what-to-expect.adoc,"Durch das Gelernte können sie auf Grundlage angemessen detaillierter Anforderungen und Randbedingungen eine adäquate Softwarearchitektur entwerfen, kommunizieren, analysieren, bewerten und weiterentwickeln.",13.0,13.0,=== What Does a Foundation Level Training Convey?,48.0,48.0,0,1
../docs\00-preamble\02-out-of-scope.adoc,Dieser Lehrplan reflektiert den aus heutiger Sicht des iSAQB e.V. notwendigen und sinnvollen Inhalt zur Erreichung der Lernziele des CPSA-F. Er stellt keine vollständige Beschreibung des Wissensgebiets „Softwarearchitektur“ dar.,5.0,5.0,This curriculum reflects the contents currently considered by the iSAQB members to be necessary and useful for achieving the learning goals of CPSA-F. It is not a comprehensive description of the entire domain of 'software architecture'.,24.0,24.0,3,2
../docs\00-preamble\03-prerequisites.adoc,Der iSAQB e. V. kann in Zertifizierungsprüfungen die hier genannten Voraussetzungen durch entsprechende Fragen prüfen.,4.0,4.0,The iSAQB e. V. may check the following prerequisites in certification examinations via corresponding questions.,38.0,38.0,2,1
../docs\00-preamble\04-structure-timing-didactics.adoc,"[cols=""<,>"", options=""header,footer""]",9.0,9.0,|===,40.0,40.0,1,0
../docs\00-preamble\05-exam-relevance-levels.adoc,Jedes Lernziel beschreibt die zu vermittelnden Inhalte inklusive ihrer Kernbegriffe und -konzepte. Bezüglich der Prüfungsrelevanz verwendet der Lehrplan folgende Kategorien:,6.0,6.0,Every learning goal describes the contents to be taught including their key terms and concepts.,41.0,41.0,0,1
../docs\00-preamble\06-original-doc.adoc,// end::DE[],11.0,11.0,// end::EN[],23.0,23.0,0,1
../docs\01-basics\01-basics-duration-terms.adoc,{glossary_url}constraint[Randbedingungen];,31.0,31.0,None,65.0,65.0,1,0
../docs\01-basics\LG-01-03.adoc,Softwarearchitekt:innen tragen die Verantwortung für die Erreichung der Anforderungen und die Entwicklung der Architektur der Lösung.,5.0,5.0,"Depending on the actual approach or process model used, they must align this responsibility with the overall project responsibility of project management or other roles.",27.0,27.0,1,0
../docs\01-basics\LG-01-04.adoc,Diese Architekturdomänen sind nicht inhaltlicher Fokus vom CPSA-F.,17.0,17.0,"* system architecture (can have various semantics, depending on the definition of ""system"")",37.0,37.0,0,1


## Phase II: Translation correction

### Prepare Dataframe

Am additional Dataframe based on the `structural_df`.  For this, we simplay drop all blank lines which don't need to be checked.

In [102]:
result_dfs = []

for filename, group in structural_df.groupby('filename'):
    # Reset index for each group to ensure correct row ordering
    group = group.reset_index(drop=True)

    # Remove NaN rows for DE section and shift the rest upwards
    non_nan_de = group.dropna(subset=['text_DE']).reset_index(drop=True)
    group['text_DE'], group['line_DE'] = non_nan_de['text_DE'], non_nan_de['line_DE']
    
    # Remove NaN rows for EN section and shift the rest upwards
    non_nan_en = group.dropna(subset=['text_EN']).reset_index(drop=True)
    group['text_EN'], group['line_EN'] = non_nan_en['text_EN'], non_nan_en['line_EN']
    
    # Append the processed group back to the result list
    result_dfs.append(group)

# Concatenate all the processed groups back together
translations_df = pd.concat(result_dfs, ignore_index=True).dropna(subset=['text_DE', 'line_DE', 'text_EN', 'line_EN'], how='all')

display(HTML(translations_df.to_html(escape=False)))

,filename,text_DE,line_DE,link_DE,text_EN,line_EN,link_EN,sentence_count_DE,sentence_count_EN
0,../docs\00-preamble\01-what-to-expect.adoc,// tag::DE[],1.0,1.0,// tag::EN[],36.0,36.0,1,1
1,../docs\00-preamble\01-what-to-expect.adoc,=== Was dieser Lehrplan enthält,2.0,2.0,=== What Does this Curriculum Contain,37.0,37.0,1,1
2,../docs\00-preamble\01-what-to-expect.adoc,"Dieser Lehrplan für den _Certified Professional for Software Architecture - Foundation Level_ (CPSA-F) beinhaltet die Lernziele, die man beherrschen sollte, um die Rolle Softwarearchitekt:in zu übernehmen.",3.0,3.0,This curriculum for the Certified Professional for Software Architecture – Foundation Level (CPSA-F) outlines the essential learning goals that should be mastered to take up the role of software architect.,38.0,38.0,1,1
3,../docs\00-preamble\01-what-to-expect.adoc,Seine Struktur orientiert sich an den grundlegenden Aktivitäten und Verantwortlichkeiten der Softwarearchitektur als Rolle:,5.0,4.0,It is structured along the fundamental activities and responsibilities of software architecture as a role:,40.0,39.0,0,0
4,../docs\00-preamble\01-what-to-expect.adoc,* Anforderungen und Randbedingungen klären,7.0,5.0,* Clarifying stakeholder requirements and constraints,42.0,40.0,1,1
5,../docs\00-preamble\01-what-to-expect.adoc,"* Entwurf und Entwicklung von Softwarearchitekturen, dabei strukturelle und konzeptionelle Entscheidungen treffen",8.0,6.0,"* Designing and developing software architectures, thereby taking structural and conceptual decisions",43.0,41.0,0,0
6,../docs\00-preamble\01-what-to-expect.adoc,* Beschreiben und Kommunizieren von Softwarearchitekturen für verschiedene Stakeholder,9.0,7.0,* Communicating and documenting the architecture for various stakeholders,44.0,42.0,1,1
7,../docs\00-preamble\01-what-to-expect.adoc,* Analysieren und Bewerten von Softwarearchitekturen,10.0,8.0,* Analyzing and assessing software architectures,45.0,43.0,1,1
8,../docs\00-preamble\01-what-to-expect.adoc,=== Was vermittelt eine Foundation-Level-Schulung?,14.0,9.0,=== What Does a Foundation Level Training Convey?,48.0,44.0,1,1
9,../docs\00-preamble\01-what-to-expect.adoc,Lizenzierte Schulungen zum _Certified Professional for Software Architecture – Foundation Level_ (CPSA-F) vermitteln grundlegende Kenntnisse und Fertigkeiten für den Entwurf einer angemessenen Softwarearchitektur für kleine und mittlere IT-Systeme.,15.0,10.0,"Licensed Certified Professional for Software Architecture – Foundation Level (CPSA-F) trainings will provide participants with the knowledge and skills required to design, specify and document a software architecture adequate to fulfil the respective requirements for small- and medium-sized systems.",49.0,45.0,1,1


In [65]:
base_df_de = base_df[base_df['lang'] == 'DE'].reset_index(drop=True)
base_df_en = base_df[base_df['lang'] == 'EN'].reset_index(drop=True)

# just keep what's not the same
structural_df = base_df_de[['filename', 'text', 'line']].join(base_df_en[['text', 'line']], lsuffix="_DE", rsuffix="_EN")
structural_df.head(10)

,filename,link,text_DE,line_DE,text_EN,line_EN
0,../docs\00-preamble\01-what-to-expect.adoc,DE EN,// tag::DE[],1.0,// tag::EN[],36.0
1,../docs\00-preamble\01-what-to-expect.adoc,DE EN,=== Was dieser Lehrplan enthält,2.0,=== What Does this Curriculum Contain,37.0
2,../docs\00-preamble\01-what-to-expect.adoc,DE EN,"Dieser Lehrplan für den _Certified Professional for Software Architecture - Foundation Level_ (CPSA-F) beinhaltet die Lernziele, die man beherrschen sollte, um die Rolle Softwarearchitekt:in zu übernehmen.",3.0,This curriculum for the Certified Professional for Software Architecture – Foundation Level (CPSA-F) outlines the essential learning goals that should be mastered to take up the role of software architect.,38.0
3,../docs\00-preamble\01-what-to-expect.adoc,DE EN,Seine Struktur orientiert sich an den grundlegenden Aktivitäten und Verantwortlichkeiten der Softwarearchitektur als Rolle:,5.0,It is structured along the fundamental activities and responsibilities of software architecture as a role:,40.0
4,../docs\00-preamble\01-what-to-expect.adoc,DE EN,* Anforderungen und Randbedingungen klären,7.0,* Clarifying stakeholder requirements and constraints,42.0
5,../docs\00-preamble\01-what-to-expect.adoc,DE EN,"* Entwurf und Entwicklung von Softwarearchitekturen, dabei strukturelle und konzeptionelle Entscheidungen treffen",8.0,"* Designing and developing software architectures, thereby taking structural and conceptual decisions",43.0
6,../docs\00-preamble\01-what-to-expect.adoc,DE EN,* Beschreiben und Kommunizieren von Softwarearchitekturen für verschiedene Stakeholder,9.0,* Communicating and documenting the architecture for various stakeholders,44.0
7,../docs\00-preamble\01-what-to-expect.adoc,DE EN,* Analysieren und Bewerten von Softwarearchitekturen,10.0,* Analyzing and assessing software architectures,45.0
8,../docs\00-preamble\01-what-to-expect.adoc,DE EN,=== Was vermittelt eine Foundation-Level-Schulung?,14.0,=== What Does a Foundation Level Training Convey?,48.0
9,../docs\00-preamble\01-what-to-expect.adoc,DE EN,Lizenzierte Schulungen zum _Certified Professional for Software Architecture – Foundation Level_ (CPSA-F) vermitteln grundlegende Kenntnisse und Fertigkeiten für den Entwurf einer angemessenen Softwarearchitektur für kleine und mittlere IT-Systeme.,15.0,"Licensed Certified Professional for Software Architecture – Foundation Level (CPSA-F) trainings will provide participants with the knowledge and skills required to design, specify and document a software architecture adequate to fulfil the respective requirements for small- and medium-sized systems.",49.0


In [27]:
translations = structural_df.dropna(subset=['text_DE', 'text_EN'], how="all")
translations.head(10)

,filename,text_DE,line_DE,text_EN,line_EN
0,../docs\00-preamble\01-what-to-expect.adoc,// tag::DE[],1,// tag::EN[],36
1,../docs\00-preamble\01-what-to-expect.adoc,=== Was dieser Lehrplan enthält,2,=== What Does this Curriculum Contain,37
2,../docs\00-preamble\01-what-to-expect.adoc,Dieser Lehrplan für den _Certified Professiona...,3,This curriculum for the Certified Professional...,38
4,../docs\00-preamble\01-what-to-expect.adoc,Seine Struktur orientiert sich an den grundleg...,5,It is structured along the fundamental activit...,40
6,../docs\00-preamble\01-what-to-expect.adoc,* Anforderungen und Randbedingungen klären,7,* Clarifying stakeholder requirements and cons...,42
7,../docs\00-preamble\01-what-to-expect.adoc,* Entwurf und Entwicklung von Softwarearchitek...,8,* Designing and developing software architectu...,43
8,../docs\00-preamble\01-what-to-expect.adoc,* Beschreiben und Kommunizieren von Softwarear...,9,* Communicating and documenting the architectu...,44
9,../docs\00-preamble\01-what-to-expect.adoc,* Analysieren und Bewerten von Softwarearchite...,10,* Analyzing and assessing software architectures,45
12,../docs\00-preamble\01-what-to-expect.adoc,NaN,13,=== What Does a Foundation Level Training Convey?,48
13,../docs\00-preamble\01-what-to-expect.adoc,=== Was vermittelt eine Foundation-Level-Schul...,14,Licensed Certified Professional for Software A...,49


### Set up OpenAI
This cell configures OpenAI's ChatGPT. The very important part is the structured response format of the request to ChatGPT.

There is also a separate file written just in case of failures of the calls to OpenAI. Technically, you could read in this file and work with it afterwards.

In [7]:
from openai import OpenAI
import os

# set the role of the LLM
ai_system = """
You are a translator for iSAQB curricula between German and English.
Your task is to identify translation errors and inconsistencies between translations."
"""

# define the schema and format including what we expect from the LLM's result
response_json_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "result",
        "schema": {
            "type": "object",
            "properties": {
                "correctness": {
                    "type": "number",
                    "description" : "a value between 0 and 1 that indicates how well the two texts fit together"},
                "assessment": {
                    "type": "string",
                    "description" : "a brief assessment of how good the translation into English is"},
                "corrected_text_de": {
                    "type": "string",
                    "description" : "an improved translation of the German text"},
                "corrected_text_en": {
                        "type": "string",
                        "description": "an improved translation of the English text"}
            },
            "required": ["correctness", "assessment", "corrected_text_de", "corrected_text_en"],
            "additionalProperties": False
        },
        "strict": True
    }
}

openai_client = OpenAI(
   api_key = os.environ["OPENAI_API_KEY"]
)

# the magic function that does all the work for us!
def correct_translation(content):

    messages=[
        {"role": "system", "content": ai_system},
        {"role": "user", "content": content}
    ]

    completion = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages = messages,
        response_format = response_json_format,
        temperature = 0,
        n=1)

    return completion.choices[0]

### Create corrected translations

This is the core of this notebook. It sends the German and the English translation to ChatGPT and requests an assessment.

Note: This might take very long for huge data. With `translations_to_correct`, you can just set a subset of the data to correct.m

In [15]:
import json


# this enables just to check a subset of the files
translations_to_correct = translations[:10].reset_index(drop=True)


cache_filepath = "llm_results.json"
# Clear the file by opening it in write mode initially
with open(cache_filepath, 'w') as json_file:
    pass  # This clears the file

results = []

# Open the JSON file in append mode
with open(cache_filepath, 'a') as json_file:
    for i, r in translations_to_correct.iterrows():
        try:
            # Define the prompt for the LLM
            prompt = f"""
            Here is the German and English translation: 

            German text:

            {r['text_DE']}

            English text:

            {r['text_EN']}

            Keep the AsciiDoc formatting as it is. Give also hints and corrections if there are differences between the German formatting and the English formatting.

            Return a result in the defined JSON schema.
            """

            # Call the translation correction function
            result = correct_translation(prompt)

            # Convert the result to a Python dictionary
            result_dict = json.loads(result.message.content)

            # Append the result to the result list
            results.append(result_dict)

            # Append the result to the JSON file
            json.dump(result_dict, json_file)
            json_file.write('\n')  # Add a new line for each result

            # Optional: Immediately flush the file to ensure the result is written
            json_file.flush()

        except Exception as e:
            # Log the error and continue with the next row
            print(f"Error occurred on row {i}: {str(e)}")

result_df = pd.DataFrame.from_dict(results)
result_df.head(5)

,correctness,assessment,corrected_text_de,corrected_text_en
0,0.0,"The provided texts are empty, making it imposs...",,
1,0.9,"The translation is mostly accurate, but the En...",=== Was dieser Lehrplan enthält,=== What this Curriculum Contains
2,0.9,"The translation is mostly accurate, but there ...",Dieser Lehrplan für den _Certified Professiona...,This curriculum for the _Certified Professiona...
3,0.9,"The translation is mostly accurate, but the ph...",Seine Struktur orientiert sich an den grundleg...,Its structure is oriented towards the fundamen...
4,0.7,The translation captures the general meaning b...,* Anforderungen und Randbedingungen klären,* Clarifying requirements and constraints


### Combine assessment result with content datam

In [16]:
assessed_content = translations_to_correct.join(result_df)[[
    'filename',
    'link',
    'text_DE',
    'corrected_text_de',
    'text_EN',
    'corrected_text_en',
    'correctness',
    'assessment',
    'line_DE',
    'line_EN']]

assessed_content.head(1)

,filename,link,text_DE,corrected_text_de,text_EN,corrected_text_en,correctness,assessment,line_DE,line_EN
0,../docs\00-preamble\01-what-to-expect.adoc,"<a href=""vscode://file/c:\dev\repos\curriculum...",// tag::DE[],,// tag::EN[],,0.0,"The provided texts are empty, making it imposs...",1,36


### Correct metadata correctness values

In [17]:
# set metadata to 1 if OK
assessed_content.loc[
    (assessed_content['text_DE'] == "// tag::DE[]") &
    (assessed_content['text_EN'] == "// tag::EN[]"), 'correctness'] = 1
assessed_content.loc[
    (assessed_content['text_DE'] == "// end::DE[]") &
    (assessed_content['text_EN'] == "// end::EN[]"), 'correctness'] = 1

### Create assessment output

In [18]:
assessed_content.to_html("translation_assessment_report.html", escape=False)
assessed_content.to_excel("translation_assessment_report.xlsx", index=None)
assessed_content.head()

,filename,link,text_DE,corrected_text_de,text_EN,corrected_text_en,correctness,assessment,line_DE,line_EN
0,../docs\00-preamble\01-what-to-expect.adoc,"<a href=""vscode://file/c:\dev\repos\curriculum...",// tag::DE[],,// tag::EN[],,1.0,"The provided texts are empty, making it imposs...",1,36
1,../docs\00-preamble\01-what-to-expect.adoc,"<a href=""vscode://file/c:\dev\repos\curriculum...",=== Was dieser Lehrplan enthält,=== Was dieser Lehrplan enthält,=== What Does this Curriculum Contain,=== What this Curriculum Contains,0.9,"The translation is mostly accurate, but the En...",2,37
2,../docs\00-preamble\01-what-to-expect.adoc,"<a href=""vscode://file/c:\dev\repos\curriculum...",Dieser Lehrplan für den _Certified Professiona...,Dieser Lehrplan für den _Certified Professiona...,This curriculum for the Certified Professional...,This curriculum for the _Certified Professiona...,0.9,"The translation is mostly accurate, but there ...",3,38
3,../docs\00-preamble\01-what-to-expect.adoc,"<a href=""vscode://file/c:\dev\repos\curriculum...",Seine Struktur orientiert sich an den grundleg...,Seine Struktur orientiert sich an den grundleg...,It is structured along the fundamental activit...,Its structure is oriented towards the fundamen...,0.9,"The translation is mostly accurate, but the ph...",5,40
4,../docs\00-preamble\01-what-to-expect.adoc,"<a href=""vscode://file/c:\dev\repos\curriculum...",* Anforderungen und Randbedingungen klären,* Anforderungen und Randbedingungen klären,* Clarifying stakeholder requirements and cons...,* Clarifying requirements and constraints,0.7,The translation captures the general meaning b...,7,42


### Set the threshold
Set the threshold for correctness. The default with 0.8 may be too ambitious.m

In [19]:
THRESHOLD = 0.8
needed_corrections = assessed_content[assessed_content['correctness'] < THRESHOLD]
display(HTML(needed_corrections.to_html(escape=False)))

,filename,link,text_DE,corrected_text_de,text_EN,corrected_text_en,correctness,assessment,line_DE,line_EN
4,../docs\00-preamble\01-what-to-expect.adoc,DE EN,* Anforderungen und Randbedingungen klären,* Anforderungen und Randbedingungen klären,* Clarifying stakeholder requirements and constraints,* Clarifying requirements and constraints,0.7,The translation captures the general meaning but introduces the term 'stakeholder' which is not present in the original German text. This could lead to a misunderstanding of the scope of the requirements being clarified.,7,42
8,../docs\00-preamble\01-what-to-expect.adoc,DE EN,=== Was vermittelt eine Foundation-Level-Schulung?,Was vermittelt eine Foundation-Level-Schulung?,"Licensed Certified Professional for Software Architecture – Foundation Level (CPSA-F) trainings will provide participants with the knowledge and skills required to design, specify and document a software architecture adequate to fulfil the respective requirements for small- and medium-sized systems.",What does a Foundation-Level training convey?,0.2,"The English translation does not accurately reflect the content of the German text. The German text asks about what a Foundation-Level training conveys, while the English text describes the training's content without addressing the question directly.",14,49
9,../docs\00-preamble\01-what-to-expect.adoc,DE EN,Lizenzierte Schulungen zum _Certified Professional for Software Architecture – Foundation Level_ (CPSA-F) vermitteln grundlegende Kenntnisse und Fertigkeiten für den Entwurf einer angemessenen Softwarearchitektur für kleine und mittlere IT-Systeme.,Lizenzierte Schulungen zum _Certified Professional for Software Architecture – Foundation Level_ (CPSA-F) vermitteln grundlegende Kenntnisse und Fertigkeiten für den Entwurf einer angemessenen Softwarearchitektur für kleine und mittlere IT-Systeme.,Based upon their individual practical experience and existing skills participants will learn to derive architectural decisions from an existing system vision and adequately detailed requirements.,Licensed training for the _Certified Professional for Software Architecture – Foundation Level_ (CPSA-F) imparts fundamental knowledge and skills for designing an appropriate software architecture for small and medium IT systems.,0.0,"The English translation does not accurately reflect the content of the German text. The German text discusses licensed training for the CPSA-F certification, focusing on fundamental knowledge and skills for designing software architecture for small and medium IT systems, while the English text talks about deriving architectural decisions from existing systems, which is unrelated.",15,50


### Produce output files



#### Create mergeable version

Produces a merge file to work with a merge editor. If you're brave, you can also set `is_override` to true to merge the suggestions of the LLM directly into the original file.

Tip: Overriding only should be enabled if both sections of the translations are almost structurally identically because otherwise it leads to complete chaos!

**Warning: Before setting it to True, make sure that you committed your local changes! Good luck!**

In [13]:
# flag for setting the override of the original file
is_override = False

# Group correctiony for each file to process each file only once
for filename, group in needed_corrections.groupby('filename'):
    # Open the file once for reading
    with open(filename, 'r') as file:
        lines = file.readlines()

    # Modify the relevant lines in memory
    modified = False
    for index, row in group.iterrows():
        if row['corrected_text_en']:  # Check if corrected_text_en is not empty
            line_number = int(row['line_EN']) - 1  # Convert to zero-based index
            # Insert merge-style corrections
            lines[line_number] = (
                f"<<<<<<< ORIGINAL\n{lines[line_number]}"
                f"=======\n{row['corrected_text_en']}\n"
                f">>>>>>> CORRECTED | German text: \'{row['text_DE']}\' (Line {int(row['line_DE'])}) | Assessment: {row['assessment']}\n"
            )
            modified = True  # Mark as modified since we are replacing the line

    # Write the updated content to a new file ending with '_merge.adoc'
    if modified:
        new_filename = filename if is_override else filename.replace('.adoc', '_merge.adoc')
        with open(new_filename, 'w') as file:
            file.writelines(lines)

        print(f"Merge-style corrected file saved as {new_filename}")

Merge-style corrected file saved as ../docs\00-preamble\01-what-to-expect.adoc
Merge-style corrected file saved as ../docs\00-preamble\02-out-of-scope.adoc
Merge-style corrected file saved as ../docs\00-preamble\03-prerequisites.adoc
Merge-style corrected file saved as ../docs\00-preamble\04-structure-timing-didactics.adoc


#### Create diffable version
Version that produces a separate file to work with a diff editor. It's disabled by default via the `is_active` flag because the mergable version is way better (IMHO).

In [14]:
# this is just a flag to set this block active or not
is_active = False

# Group by 'filename' to process each file only once
for filename, group in needed_corrections.groupby('filename'):
    # Open the file once for reading
    with open(filename, 'r') as file:
        lines = file.readlines()

    # Modify the relevant lines in memory
    modified = False
    for index, row in group.iterrows():
        line_number = int(row['line_EN']) - 1  # Convert to zero-based index
        lines[line_number] = row['corrected_text_en'] + '\n'
        modified = True  # Mark as modified since we are replacing the line

    # Write the updated content to a new file ending with '_corrected.adoc'
    if modified and is_active:
        new_filename = filename.replace('.adoc', '_aicorrected.adoc')
        with open(new_filename, 'w') as file:
            file.writelines(lines)

## Summary

Happy correcting!

Markus Harrer, October 2024